### Details
**Name:** Tharun Otturu  
**Student Id:** IITP_AIMLT_2601782

### Task 1: Creating the database

In [1]:
import random
import sqlite3
import pandas as pd

In [2]:
## Defining a function that creates the employee database and returns the cursor and the connection.
def create_employee_database():
    conn = sqlite3.connect('employees.db')

    cursor = conn.cursor()

    cursor.execute(
    '''
        CREATE TABLE IF NOT EXISTS employees (
            emp_id INTEGER PRIMARY KEY,
            name TEXT,
            department TEXT,
            salary REAL,
            years_experience INTEGER,
            performance_score REAL
        )
    '''
    )

    return cursor, conn


In [3]:
## Generating records inside a function
def generate_employee_records(num_records):
    records = []
    for i in range(num_records):
        emp_id = i + 1
        name = f"Employee_{i+1}"
        department = random.choice(departments)
        salary = round(random.uniform(50000, 150000), 2)
        years_experience = random.randint(1,15)
        performance_score = random.uniform(1,5)

        employee = (emp_id, name, department, salary, years_experience, performance_score)
        records.append(employee)

    return records


In [4]:
## Creating and Inserting the Employee records:
departments       = ["Engineering", "Sales", "Marketing", "HR", "Finance"]
emp_records       = generate_employee_records(50)
cursor, conn      = create_employee_database()


cursor.executemany('''
    INSERT OR IGNORE INTO employees (emp_id, name, department, salary, years_experience, performance_score)
    VALUES (?, ?, ?, ?, ?, ?)
''', emp_records)

conn.commit()

### Task 2: SQL Queries

In [5]:
q1 = '''
    SELECT name, department, salary, performance_score from employees
    WHERE performance_score >= 4 AND years_experience >= 3
    ORDER BY performance_score DESC
    LIMIT 15
'''

q2 = '''
    SELECT * FROM employees
    WHERE salary BETWEEN 70000 AND 110000
    AND department IN ('Engineering', 'Sales')
    ORDER BY department, salary DESC
'''

q3 = '''
    SELECT department, COUNT(emp_id) as [number_of_employees], AVG(salary) as [average_salary] FROM employees
    GROUP BY department
    ORDER BY average_salary DESC
'''

q4 = '''
    SELECT * FROM employees
'''



In [6]:
d1 = pd.read_sql_query(q1, conn)
print(d1.to_string())

d2 = pd.read_sql_query(q2, conn)
print(d2.to_string())

d3 = pd.read_sql_query(q3, conn)
print(d3.to_string())


           name   department     salary  performance_score
0   Employee_29  Engineering  102188.59           4.987916
1   Employee_37  Engineering  132000.95           4.900014
2   Employee_41  Engineering   84930.57           4.818322
3   Employee_10  Engineering  135103.18           4.774256
4   Employee_17    Marketing  146437.81           4.741179
5   Employee_66  Engineering  123089.06           4.694753
6    Employee_2           HR  149116.81           4.629148
7   Employee_22      Finance  139138.49           4.508104
8   Employee_54    Marketing   72976.75           4.506258
9    Employee_5  Engineering   97775.53           4.499360
10  Employee_87  Engineering   71922.35           4.485022
11  Employee_13           HR  146631.71           4.438139
12  Employee_60    Marketing  102217.28           4.358013
13  Employee_74      Finance  128848.86           4.290027
14  Employee_39    Marketing   96346.05           4.279700
    emp_id         name   department     salary  years_e

### Task 3: Pandas Analysis

In [7]:
q4 = '''
    SELECT * FROM employees
'''


employees_df = pd.read_sql_query(q4, conn)
print(employees_df.to_string())


    emp_id          name   department     salary  years_experience  performance_score
0        1    Employee_1      Finance   63258.96                 5           1.949307
1        2    Employee_2           HR  149116.81                10           4.629148
2        3    Employee_3      Finance  128277.04                 8           3.923246
3        4    Employee_4           HR  140646.39                 2           2.822964
4        5    Employee_5  Engineering   97775.53                15           4.499360
5        6    Employee_6           HR  147837.30                 3           1.240311
6        7    Employee_7  Engineering  100134.68                 2           4.127253
7        8    Employee_8  Engineering  138897.99                11           1.846615
8        9    Employee_9           HR  136706.39                12           2.488165
9       10   Employee_10  Engineering  135103.18                 6           4.774256
10      11   Employee_11        Sales  141527.45      

#### Replicating Query 1 in pandas

In [8]:
pq1 = employees_df.loc[
    (employees_df["performance_score"] >= 4) & (employees_df["years_experience"] >= 3),
    ["name", "department", "salary", "performance_score"]
].sort_values(by="performance_score", ascending=False).head(15)

## Verifying both the dataframes are equal
print(pq1.reset_index(drop=True).equals(d1.reset_index(drop=True)))


True


#### SQL vs Pandas

1. Both SQL and pandas are very good tools to store and analyse data, but their syntax and usecases are different.
2. While SQL is used to store very large data, pandas is used to work on relatively smaller data sets. This is because pandas loads the data into the memory before it is ready for any operations. Whereas SQL stores all data on disk.
3. There are also certain syntactic differences like: pandas uses &/==/sort_values whereas SQL uses AND/=/ORDER BY. SQL is designed to be more user-friendly, making its syntax relatively simple and easy to understand.
4. General recomendation is to use SQL to store large data. And then when it comes to data analysis for insights, load filtered data into memory and use pandas for the analysis. This approach utilizes the strengths of both the tools effectively.